# 🔧 LoRA 파인튜닝

## LoRA (Low-Rank Adaptation)란?

LoRA는 대형 언어 모델(LLM)을 효율적으로 미세조정하는 기법입니다.

### 핵심 원리
- 사전 학습된 가중치 행렬 **W**에 저차원(low-rank) 분해 행렬 **ΔW = BA**를 더합니다
- 원래 가중치 W는 고정(freeze)하고, A와 B만 학습합니다
- 전체 파라미터의 **1% 미만**만 학습하면서도 우수한 성능을 달성합니다

### 왜 LoRA를 사용하나?
| 장점 | 설명 |
|------|------|
| **메모리 효율** | 전체 모델 대비 훨씬 적은 GPU 메모리 사용 |
| **빠른 학습** | 학습 파라미터가 적어 수렴이 빠름 |
| **모듈성** | 어댑터만 교체하여 다양한 도메인 적용 가능 |
| **원본 보존** | 기본 모델 가중치를 변경하지 않음 |

### 이 노트북의 설정
- **기본 모델**: `Qwen/Qwen3-4B-Instruct-2507`
- **LoRA rank (r)**: 16, **alpha**: 32
- **대상 모듈**: attention (Q/K/V/O) + MLP (gate/up/down)
- **손실 마스킹**: assistant 응답만 학습 (시스템/사용자 메시지는 제외)

> ⚠️ 이 노트북은 **준비된 데이터 번들**을 사용합니다.  
> SDG, 교사 모델 호출, 데이터 수정은 하지 않습니다.  
> 번들이 유효하지 않으면 학습이 중단됩니다.

In [ ]:
"""환경 확인 — 00_preflight.ipynb 에서 이미 설치 완료."""

import os
from pathlib import Path

_nb_dir = Path.cwd()
_project_root = _nb_dir
for _p in [_nb_dir] + list(_nb_dir.parents):
    if (_p / "pyproject.toml").exists():
        _project_root = _p
        break

os.environ["RHOAI_PROJECT_ROOT"] = str(_project_root)

try:
    import rhoai_model_training_lab.config as _cfg
    _cfg.PROJECT_ROOT = _project_root
    print(f"✅ 프로젝트 루트: {_project_root}")
except ImportError:
    raise ImportError(
        "❌ 패키지 미설치 — 먼저 00_preflight.ipynb 를 실행하세요."
    )


In [ ]:
"""Load configuration and validate bundle compatibility."""

import os
from pathlib import Path

from rhoai_model_training_lab.config import (
    load_env, load_training_config, load_bundle_config, PROJECT_ROOT,
)
from rhoai_model_training_lab.data import BundleManager

load_env()

# Load LoRA config
lora_config = load_training_config("lora")
model_id = lora_config["model"]["model_id"]
model_revision = lora_config["model"]["model_revision"]

print(f"모델: {model_id} (rev: {model_revision})")
print(f"LoRA r={lora_config['lora']['r']}, alpha={lora_config['lora']['lora_alpha']}")
print(f"대상 모듈: {lora_config['lora']['target_modules']}")
print(f"시드: {lora_config['training']['seed']}")

# Load and validate bundle
release_config = load_bundle_config()
bundle_base = release_config.get("bundle", {}).get("base_path", "data/prepared/tau-knowledge-v1")
bundle_path = PROJECT_ROOT / bundle_base

print(f"\n번들 경로: {bundle_path}")
mgr = BundleManager.load_bundle(bundle_path)
manifest = mgr.manifest

# Compatibility check
compat = mgr.validate_compatibility(model_id)
if compat.errors:
    for err in compat.errors:
        print(f"  ❌ {err}")
    raise RuntimeError(
        "번들 호환성 검증 실패 — 학습을 진행할 수 없습니다.\n"
        "data_preparation/ 노트북에서 올바른 번들을 생성하세요."
    )

print(f"\n✅ 번들 호환성 검증 통과")
print(f"   학습 샘플: {manifest.canonical_train_count}")
print(f"   검증 샘플: {manifest.canonical_validation_count}")
print(f"   번들 버전: {manifest.bundle_version}")

In [ ]:
"""Preview training data with loss mask visualization."""

from transformers import AutoTokenizer

# Load tokenizer for visualization
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Load backend-specific training data
train_data = mgr.get_training_samples("lora", "train")
val_data = mgr.get_training_samples("lora", "validation")

print(f"LoRA 학습 데이터: {len(train_data)} 샘플")
print(f"LoRA 검증 데이터: {len(val_data)} 샘플")

# Preview first 2 examples with loss mask visualization
print("\n" + "=" * 70)
print("📝 학습 데이터 미리보기 (손실 마스크 시각화)")
print("=" * 70)

for i, sample in enumerate(train_data[:2]):
    messages = sample.get("messages", [])
    print(f"\n--- 샘플 {i+1} ---")
    print(f"메시지 수: {len(messages)}")

    for msg in messages:
        role = msg["role"]
        content = msg.get("content", "") or ""
        # Loss mask: only assistant messages contribute to loss
        is_target = role == "assistant"
        mask_icon = "🎯 [학습 대상]" if is_target else "🚫 [마스킹됨]"

        display_content = content[:200] + "..." if len(content) > 200 else content
        print(f"\n  {mask_icon} [{role}]:")
        print(f"    {display_content}")

        if msg.get("tool_calls"):
            print(f"    🔧 도구 호출: {len(msg['tool_calls'])}건")
            for tc in msg["tool_calls"][:2]:
                fn = tc.get("function", {})
                print(f"       → {fn.get('name', '?')}({fn.get('arguments', '')[:80]})")

    # Token count
    try:
        formatted = tokenizer.apply_chat_template(messages, tokenize=True)
        print(f"\n  토큰 수: {len(formatted)}")
        max_len = lora_config["data"]["max_seq_length"]
        if len(formatted) > max_len:
            print(f"  ⚠️  최대 길이 {max_len} 초과!")
    except Exception:
        pass

print(f"\n{'=' * 70}")
print("손실 마스킹 정책: assistant 응답만 학습 대상")
print("시스템 메시지, 사용자 입력, 도구 관찰값은 손실 계산에서 제외됩니다.")

In [ ]:
"""training_hub.lora_sft 로 LoRA 학습 실행."""

import time
import subprocess, sys
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU가 감지되지 않았습니다.\n"
        "LoRA 학습에는 CUDA 호환 GPU가 필요합니다."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
print()

# 파일 경로 준비
train_file = str(PROJECT_ROOT / lora_config["data"]["train_file"])
val_file = str(PROJECT_ROOT / lora_config["data"]["validation_file"])
output_dir = str(PROJECT_ROOT / lora_config["training_args"]["output_dir"])

print("=" * 70)
print("🚀 LoRA 학습 시작")
print("=" * 70)
print(f"  모델: {model_id}")
print(f"  학습 데이터: {train_file}")
print(f"  검증 데이터: {val_file}")
print(f"  출력: {output_dir}")
print(f"  에포크: {lora_config['training_args']['num_train_epochs']}")
print(f"  배치: {lora_config['training_args']['per_device_train_batch_size']} x {lora_config['training_args']['gradient_accumulation_steps']} (누적)")
print(f"  학습률: {lora_config['training_args']['learning_rate']}")
print(f"  LoRA r={lora_config['lora']['r']}, alpha={lora_config['lora']['lora_alpha']}")

# 백엔드 선택: unsloth → huggingface 폴백
backend = "unsloth"
try:
    import unsloth  # noqa: F401
    print(f"  백엔드: unsloth ✅")
except ImportError:
    backend = "huggingface"
    print(f"  백엔드: huggingface (unsloth 미설치 → 자동 폴백)")
print()

# MLflow 설정
mlflow_uri = os.environ.get("MLFLOW_TRACKING_URI", "")
mlflow_experiment = os.environ.get("MLFLOW_EXPERIMENT_TRAINING", "rhoai-model-training-lab-training")

start_time = time.time()

from training_hub import lora_sft

training_result = lora_sft(
    model_path=model_id,
    data_path=train_file,
    ckpt_output_dir=output_dir,
    backend=backend,

    # LoRA
    lora_r=lora_config["lora"]["r"],
    lora_alpha=lora_config["lora"]["lora_alpha"],
    lora_dropout=lora_config["lora"]["lora_dropout"],
    target_modules=lora_config["lora"]["target_modules"],

    # Training
    num_epochs=lora_config["training_args"]["num_train_epochs"],
    micro_batch_size=lora_config["training_args"]["per_device_train_batch_size"],
    gradient_accumulation_steps=lora_config["training_args"]["gradient_accumulation_steps"],
    learning_rate=lora_config["training_args"]["learning_rate"],
    max_seq_len=lora_config["data"]["max_seq_length"],
    lr_scheduler=lora_config["training_args"]["lr_scheduler_type"],
    bf16=lora_config["training_args"]["bf16"],
    flash_attention=True,
    trust_remote_code=True,

    # Logging & Saving
    logging_steps=lora_config["training_args"]["logging_steps"],
    eval_steps=lora_config["training_args"]["eval_steps"],
    save_steps=lora_config["training_args"]["save_steps"],
    save_total_limit=lora_config["training_args"]["save_total_limit"],

    # Evaluation
    eval_data_path=val_file,

    # MLflow (built-in)
    mlflow_tracking_uri=mlflow_uri or None,
    mlflow_experiment_name=mlflow_experiment if mlflow_uri else None,
    mlflow_run_name=f"lora-{manifest.bundle_version}" if mlflow_uri else None,

    # Dataset format
    dataset_type="chat",
    field_messages="messages",
)

wall_time = time.time() - start_time

print(f"\n✅ LoRA 학습 완료!")
print(f"  소요 시간: {wall_time/60:.1f}분")
print(f"  피크 VRAM: {torch.cuda.max_memory_allocated() / (1024**3):.1f} GB")


In [ ]:
"""학습 결과 확인 — 손실 곡선 및 메트릭."""

import json
from pathlib import Path

output_dir_path = Path(output_dir)

# training_hub 반환값에서 trainer 추출
trainer = None
if isinstance(training_result, dict):
    trainer = training_result.get("trainer")

# 로그 히스토리 로드
log_history = None
if trainer and hasattr(trainer, "state") and hasattr(trainer.state, "log_history"):
    log_history = trainer.state.log_history

if log_history is None:
    state_path = output_dir_path / "trainer_state.json"
    if state_path.exists():
        with open(state_path) as f:
            log_history = json.load(f).get("log_history", [])

if log_history:
    train_steps = [e["step"] for e in log_history if "loss" in e]
    train_losses = [e["loss"] for e in log_history if "loss" in e]
    eval_steps = [e["step"] for e in log_history if "eval_loss" in e]
    eval_losses = [e["eval_loss"] for e in log_history if "eval_loss" in e]

    try:
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(1, 1, figsize=(10, 5))
        ax.plot(train_steps, train_losses, label="학습 손실", alpha=0.7)
        if eval_losses:
            ax.plot(eval_steps, eval_losses, label="검증 손실", marker="o", markersize=4)
        ax.set_xlabel("Step")
        ax.set_ylabel("Loss")
        ax.set_title("LoRA 학습 손실 곡선")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("학습 손실 추이 (최근 10 스텝):")
        for step, loss in zip(train_steps[-10:], train_losses[-10:]):
            bar = "█" * int(loss * 20)
            print(f"  Step {step:>6}: {loss:.4f} {bar}")

    print("\n--- 학습 메트릭 요약 ---")
    if train_losses:
        print(f"  초기 손실: {train_losses[0]:.4f}")
        print(f"  최종 손실: {train_losses[-1]:.4f}")
        print(f"  최소 손실: {min(train_losses):.4f}")
    if eval_losses:
        print(f"  최종 검증 손실: {eval_losses[-1]:.4f}")
        print(f"  최소 검증 손실: {min(eval_losses):.4f}")

    peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
    print(f"\n  피크 VRAM: {peak_vram:.1f} GB")
    print(f"  총 학습 스텝: {train_steps[-1] if train_steps else 'N/A'}")
else:
    print("⚠️  학습 로그를 찾을 수 없습니다.")
    train_losses, eval_losses, train_steps, eval_steps = [], [], [], []


In [ ]:
"""학습 결과 로컬 저장."""

# training_hub의 mlflow_tracking_uri 옵션으로 이미 MLflow에 기록됨
# 여기서는 로컬 결과 요약 저장

result_summary = {
    "method": "lora",
    "model_id": model_id,
    "bundle_name": manifest.bundle_name,
    "bundle_version": manifest.bundle_version,
    "train_samples": manifest.canonical_train_count,
    "validation_samples": manifest.canonical_validation_count,
    "lora_r": lora_config["lora"]["r"],
    "lora_alpha": lora_config["lora"]["lora_alpha"],
    "learning_rate": lora_config["training_args"]["learning_rate"],
    "num_epochs": lora_config["training_args"]["num_train_epochs"],
    "wall_time_seconds": wall_time,
    "peak_vram_gb": torch.cuda.max_memory_allocated() / (1024**3),
    "gpu_name": torch.cuda.get_device_name(0),
    "final_train_loss": train_losses[-1] if train_losses else None,
    "final_eval_loss": eval_losses[-1] if eval_losses else None,
    "best_eval_loss": min(eval_losses) if eval_losses else None,
}

result_path = Path(output_dir) / "training_result.json"
result_path.parent.mkdir(parents=True, exist_ok=True)
with open(result_path, "w") as f:
    json.dump(result_summary, f, indent=2, ensure_ascii=False)
print(f"📄 학습 결과 저장: {result_path}")

if mlflow_uri:
    print(f"📊 MLflow 기록: {mlflow_uri} / {mlflow_experiment}")
else:
    print("ℹ️  MLflow 미설정 — 로컬 결과만 저장됨")


In [ ]:
"""Verify checkpoint and proceed to export."""

from rich.table import Table
from rich.console import Console

console = Console()

# Verify checkpoint files exist
checkpoint_dir = Path(output_dir)
adapter_config = checkpoint_dir / "adapter_config.json"
adapter_model = checkpoint_dir / "adapter_model.safetensors"

# Find the best/latest checkpoint
checkpoints = sorted(checkpoint_dir.glob("checkpoint-*"), key=lambda p: p.name)
final_adapter = checkpoint_dir  # training_hub may save directly to output_dir

checks = []

# Check adapter config
has_config = adapter_config.exists() or any(
    (cp / "adapter_config.json").exists() for cp in checkpoints
)
checks.append(("어댑터 설정 파일", has_config))

# Check adapter weights
has_weights = adapter_model.exists() or any(
    list((cp).glob("adapter_model*")) for cp in checkpoints
)
checks.append(("어댑터 가중치 파일", has_weights))

# Check tokenizer
has_tokenizer = (checkpoint_dir / "tokenizer_config.json").exists() or any(
    (cp / "tokenizer_config.json").exists() for cp in checkpoints
)
checks.append(("토크나이저 파일", has_tokenizer))

# Verify reload
reload_ok = False
try:
    from peft import PeftModel, PeftConfig

    # Try loading adapter config
    adapter_source = str(checkpoint_dir)
    if not adapter_config.exists() and checkpoints:
        adapter_source = str(checkpoints[-1])

    peft_config = PeftConfig.from_pretrained(adapter_source)
    reload_ok = True
    print(f"\n✅ 어댑터 설정 리로드 성공")
    print(f"   기본 모델: {peft_config.base_model_name_or_path}")
    print(f"   r={peft_config.r}, alpha={peft_config.lora_alpha}")
except Exception as exc:
    print(f"\n⚠️  어댑터 리로드 확인 실패: {exc}")

checks.append(("어댑터 리로드 검증", reload_ok))

# Summary table
table = Table(title="🔍 LoRA 체크포인트 검증", show_header=True)
table.add_column("항목", style="bold")
table.add_column("상태")

for name, ok in checks:
    status = "✅" if ok else "❌"
    table.add_row(name, status)

console.print(table)

all_ok = all(ok for _, ok in checks)
if all_ok:
    print("\n🎉 LoRA 학습이 성공적으로 완료되었습니다!")
    print("\n다음 단계:")
    print("  📓 04_osft_finetuning.ipynb — OSFT 학습 (독립적, 동일 데이터)")
    print("  📓 05_export_and_deploy.ipynb — 모델 내보내기 및 배포")
else:
    print("\n⚠️  일부 검증 항목이 실패했습니다. 위 결과를 확인하세요.")